# meeting_extracted_w1_t1_clean — W1 第二支：T1-b 分段 → token 化 → 分類 → 段層去重

- 用途：把展示表 `body_display` 切成「一列一段」、把人名 / email 換成參與者表的 token、判 `content_type`、算 `content_hash` / `dup_of`，寫清洗表給 T2（L1 萃取）。全是規則，無 LLM。規格 §4.1 (b) 4–7、(c)、(d)、(e)；偏離處見 `meeting_extracted_migration/STATUS.md` D13、D16–D18。
- 輸入 table：`s_meeting_record_mail_{unit}`（`body_display` 與標頭欄）、`s_meeting_participants_{unit}`（token 對照）、`s_semantic_entity_alias` / `s_semantic_entity`（`customer_hint` / `app_hint`）
- 輸出 table：`s_meeting_record_clean_{unit}`（本批 pk 先 DELETE 再 append；鍵 `pk_seq, segment_no, sub_no`）
- 參數（widgets）：`catalog`、`schema`、`run_mode`（incremental / backfill）、`date_from` / `date_to`（backfill 用，對 `mail_date`）、`unit`、`cleaner_version`、`sample_limit`、`dry_run`
- 排程：W1 每週一，接在 `meeting_extracted_w1_t1_mail` 之後（同一個 Job 的第 2 個 task）。incremental 不用水位參數：mail 表有、清洗表沒有（或 mail 表較新、或清洗表版本不同）的 pk 就是本批。
- 負責人 / 更新日期：（填）/ 2026-10-15

| cell | 做什麼 |
|---|---|
| [c03] | 與第一支共用的純函式（標頭區塊偵測、日期解析、normalize）——**複製自 `w1_t1_mail` [c02] / [c04]，改一邊要同步另一邊** |
| [c04] | 分段：thread 以標頭區塊（含前面的 `-----Original…-----` 行）為界；紀要以 `ITEM` / 型號行為界；切不開 → 段 0；記 `display_start / end`、`segment_date` |
| [c05] | 每段清洗：標頭改寫（`<P02>:` ＋ Sent 日期，To / Cc / Bcc / When / Where 刪）、**token 化**（姓名三種寫法、email、工號、帳號）、短行攤平、`content_type`、`parties` / `customer_hint` / `app_hint`、`content_hash` |
| [c06] | Spark 接線：participants 表依 pk 聚成 array 餵 UDF、`dup_of` 視窗、殘留 PII 計數 |
| [c12] | 關卡：**殘留參與者姓名 > 0**、`dup_of` > 80% 會停；本批 0 段只印（D34）；參與者表沒有的 email 已遮成 `[email]`（只計數）；其餘印 log |

隱私：清洗表 `clean_text` 不得有姓名 / email / 工號（S3 只留在展示表與參與者表）；[c12] 與 [c30] 都檢查。dry_run 樣本只印 token 化後的前 200 字。

In [ ]:
# [c01] params
# W1 第二支：T1-b 分段 → token 化 → 分類 → 段層去重，寫 s_meeting_record_clean_{unit}。無 LLM。
# 由 job parameters 覆蓋；widget 建立過後改預設值不會生效，要重設先 dbutils.widgets.removeAll()。
from datetime import date, datetime

dbutils.widgets.text("catalog", "micenter")
dbutils.widgets.text("schema", "mi3_datahub_prod")
dbutils.widgets.dropdown("run_mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.text("date_from", "2026-06-01")   # backfill：對 mail 表 mail_date，含
dbutils.widgets.text("date_to", "2026-12-31")
dbutils.widgets.text("unit", "bu")                 # 決定讀寫哪一組表（s_meeting_record_mail_{unit} …）
dbutils.widgets.text("cleaner_version", "0.1")     # 改 [c04] / [c05] 規則就升；舊版本的段會在 incremental 被重洗
dbutils.widgets.text("sample_limit", "0")          # 開發用：> 0 只取最新 N 封；正式跑 0
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"])  # true：算完印關卡與樣本，不寫表


def _date(s: str) -> date:
    return datetime.strptime(s.strip(), "%Y-%m-%d").date()


cfg = {
    "catalog": dbutils.widgets.get("catalog").strip(),
    "schema": dbutils.widgets.get("schema").strip(),
    "run_mode": dbutils.widgets.get("run_mode").strip(),
    "date_from": _date(dbutils.widgets.get("date_from")),
    "date_to": _date(dbutils.widgets.get("date_to")),
    "unit": dbutils.widgets.get("unit").strip(),
    "cleaner_version": dbutils.widgets.get("cleaner_version").strip(),
    "sample_limit": int(dbutils.widgets.get("sample_limit")),
    "dry_run": dbutils.widgets.get("dry_run") == "true",
}
assert cfg["catalog"] and cfg["schema"], "catalog / schema 不可為空"
assert cfg["date_from"] <= cfg["date_to"], "date_from 不可晚於 date_to"
assert cfg["unit"] and cfg["cleaner_version"], "unit / cleaner_version 不可為空"
print(cfg)

In [ ]:
# [c02] imports
# [c02]～[c05] 是純 Python（不碰 spark / dbutils），tests/ 直接載入；[c06] 是 Spark 接線。
import hashlib
import json
import re
import unicodedata
from datetime import date, datetime

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, IntegerType, StructField, StructType

# ---- 與 w1_t1_mail [c02] 相同的常數（改一邊要同步另一邊）----
HEADER_KEYS = ("From", "Sent", "To", "Cc", "Bcc", "Subject", "When", "Where")
KEY_RE = re.compile(
    r"\*\*(From|Sent|To|Cc|Bcc|Subject|When|Where):\*\*[ \t]*"
    r"|^[ \t]*(From|Sent|To|Cc|Bcc|Subject|When|Where):[ \t]*",
    re.M,
)
LINE_END_KEYS = {"From", "Sent", "Subject", "Where"}
ORIG_SEP_RE = re.compile(r"^[ \t]*-{3,}\s*Original (?:Message|Appointment)\s*-{3,}[ \t]*$", re.M)
MONTHS = {m: i for i, m in enumerate(
    ["january", "february", "march", "april", "may", "june", "july", "august", "september",
     "october", "november", "december"], start=1)}
EN_DT_RE = re.compile(
    r"(?:[A-Za-z]+,\s*)?([A-Za-z]+)\s+(\d{1,2}),\s*(\d{4})(?:\s+(\d{1,2}):(\d{2})(?::(\d{2}))?\s*([AP]M)?)?", re.I
)
ZH_DT_RE = re.compile(r"(\d{4})年(\d{1,2})月(\d{1,2})日(?:[^\d]{0,12}?(上午|下午)?\s*(\d{1,2}):(\d{2}))?")

# ---- T1-b 專用 ----
MIN_SEG_CHARS = 150        # < 150 → noise；不參與去重（§4.1 (d)(e)）
THREAD_MIN_CHARS = 300     # thread_reply / appointment / attachment_only 門檻
MINUTES_MIN_CHARS = 800    # 「meeting minutes」字樣要配這個長度才算 inline_minutes
SUBNO_CHARS = 12_000       # §11.1：Phase 1 不切 sub_no，只在 log 計數
SHORT_LINE_CHARS = 20      # 規則 6：攤平表格的「短行」
FLATTEN_MIN_LINES = 8      # 規則 6：連續短行 ≥ 8 行才攤
MODEL_RE = re.compile(r"\b([BMT]\d{3}[A-Z]{3}\d{2}\.[A-Z0-9]+)")       # 型號（任意位置）
MODEL_LINE_RE = re.compile(r"^[ \t]*([BMT]\d{3}[A-Z]{3}\d{2}\.[A-Z0-9]+)", re.M)  # 行首型號 = 紀要段界
ITEM_RE = re.compile(r"^[ \t]*ITEM[ \t]*:?[ \t]*$", re.M | re.I)
ITEM_DATE_RE = re.compile(r"(?:(\d{4})[/年])?(\d{1,2})[/月](\d{1,2})日?")
MINUTES_HINT_RE = re.compile(r"meeting minutes|會議紀錄|會議記錄|紀要|纪要", re.I)
ATTACH_HINT_RE = re.compile(r"see attached|as attached|attached file|如附件|請參考附件|附件|file server|please refer the attached", re.I)
TABLE_SEP_CELL_RE = re.compile(r":?-{3,}:?")
TABLE_JUNK_CELL_RE = re.compile(r"(?:https?://\S+|\[url\]|\[image\]|!\[[^\]]*\]\([^)]*\))")
TOKEN_NUM_RE = re.compile(r"<([PG])\d{2,}>")  # content_hash 前抹掉 token 編號（D21）
MD_IMG_URL_RE = re.compile(r"!\[[^\]]*\]\(https?://[^)]*\)")   # 外連圖片 → [image]
URL_RE = re.compile(r"https?://[^\s|)>\]]+")                    # 裸網址 → [url]（D21）
EMAIL_ANY_RE = re.compile(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9\-]+(?:\.[A-Za-z0-9\-]+)+")
EMP_ID_RE = re.compile(r"(?<!\d)\d{7}(?!\d)")  # AUO 工號 7 位（mail_from 第 1 段）；只刪參與者表有的那些
SIZE_HZ_RE = re.compile(r"(\d{2}(?:\.\d)?)\s*[\"”吋]\s*[^\n]{0,30}?\d{2,3}\s*Hz", re.I)
SIZE_RE = re.compile(r"(\d{2}(?:\.\d)?)\s*[\"”吋]")
MNT_SIZES = {"23.8", "24", "25", "27", "31.5", "32", "34", "37.5", "49"}
TV_SIZES = {"43", "50", "55", "58", "65", "75", "85", "98"}
NB_SIZES = {"11.6", "13.3", "14", "15.6", "16", "17.3", "18"}
SUBJECT_APP_RE = {
    "MNT": re.compile(r"\bMNT\b|monitor|顯示器|显示器", re.I),
    "NB": re.compile(r"\bNB\b|notebook|筆電|笔电", re.I),
    "TV": re.compile(r"\bTV\b|電視|电视", re.I),
}
PREFIX_APP = {"B": "NB", "M": "MNT", "T": "TV"}
CONTENT_TYPES = ("inline_minutes", "thread_reply", "appointment", "attachment_only", "noise")

# 從 mail 表讀的欄（UDF 輸入）；參與者表每列取的欄（聚成 array 餵 UDF）
MAIL_COLS = [
    "pk_seq", "drupal_id", "drupal_link", "unit", "create_date", "mail_date", "mail_datetime",
    "subject_raw", "subject_clean", "forwarder_token", "forwarder_dept_code", "meeting_when", "meeting_where",
    "body_source_field", "body_conflict", "body_display", "is_fw", "attach_count",
]
PART_COLS = ["segment_no", "token", "role", "kind", "display_name_raw", "name_zh", "name_en",
             "email", "email_domain", "emp_id", "account"]

In [ ]:
# [c03] pure_shared
# 與 w1_t1_mail [c03] / [c04] 相同的純函式（norm_space、normalize_text、sha1_hex、find_header_blocks、parse_datetime）。
# 複製而不是 import：Databricks 只能貼 cell。改一邊要同步另一邊；tests 有一條比對兩支的標頭偵測結果一致。
def norm_space(s: str | None) -> str:
    """所有空白（含 U+00A0、換行）壓成一格並去頭尾。"""
    return re.sub(r"\s+", " ", (s or "").replace("\xa0", " ")).strip()


def normalize_text(s: str | None) -> str:
    """去重 / 穩定鍵共用的正規化：NFKC（全半形統一）、去標點、去空白、去控制字元、小寫。"""
    t = unicodedata.normalize("NFKC", s or "")
    return "".join(ch for ch in t if unicodedata.category(ch)[0] not in "PZC").lower()


def sha1_hex(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8")).hexdigest()


def _last_value_end(text: str, pos: int, limit: int, key: str) -> int:
    """區塊最後一個鍵的值到哪裡：From / Sent / Subject / Where 到行尾；To / Cc / Bcc / When 到第一個空白行。"""
    if key in LINE_END_KEYS:
        nl = text.find("\n", pos)
        return limit if nl == -1 else min(nl, limit)
    m = re.compile(r"\n[ \t]*\n").search(text, pos, limit)
    return m.start() if m else limit


def find_header_blocks(text: str) -> list[dict]:
    """找出正文裡的轉寄 / 回覆標頭區塊，依文件順序（上 → 下）。
    區塊 = 一個 From 標記起、到下一個 From 標記前的 Sent / To / Cc / Bcc / Subject / When / Where；
    只有 From 沒有 Sent 也沒有 Subject 的不算（避免正文裡的「From:」）。
    回傳 [{"start", "end", "fields": {key: 值（空白已壓）}}]；start / end 是在 text 的字元偏移。"""
    marks = [(m.start(), m.end(), m.group(1) or m.group(2)) for m in KEY_RE.finditer(text)]
    blocks, i = [], 0
    while i < len(marks):
        if marks[i][2] != "From":
            i += 1
            continue
        j = i + 1
        while j < len(marks) and marks[j][2] != "From":
            j += 1
        group = marks[i:j]
        limit = marks[j][0] if j < len(marks) else len(text)
        fields, end = {}, group[0][1]
        for k, (_, me, key) in enumerate(group):
            ve = group[k + 1][0] if k + 1 < len(group) else _last_value_end(text, me, limit, key)
            fields.setdefault(key, norm_space(text[me:ve]))
            end = ve
        if "Sent" in fields or "Subject" in fields:
            blocks.append({"start": group[0][0], "end": end, "fields": fields})
        i = j
    return blocks


def parse_datetime(s: str | None) -> datetime | None:
    """Sent: / When: 的日期時間。英文長格式（星期可有可無、秒可有可無、AM/PM）與中文 `yyyy年m月d日 上午/下午 h:mm`。
    只取第一個時間（When 的區間取開始）。解析不到 → None。"""
    t = norm_space(s)
    if not t:
        return None
    m = EN_DT_RE.search(t)
    if m and m.group(1).lower() in MONTHS:
        mon, day, year = MONTHS[m.group(1).lower()], int(m.group(2)), int(m.group(3))
        hh, mm = int(m.group(4) or 0), int(m.group(5) or 0)
        ap = (m.group(7) or "").upper()
        if ap == "PM" and hh < 12:
            hh += 12
        if ap == "AM" and hh == 12:
            hh = 0
        try:
            return datetime(year, mon, day, hh, mm)
        except ValueError:
            return None
    m = ZH_DT_RE.search(t)
    if m:
        year, mon, day = int(m.group(1)), int(m.group(2)), int(m.group(3))
        hh, mm = int(m.group(5) or 0), int(m.group(6) or 0)
        if m.group(4) == "下午" and hh < 12:
            hh += 12
        try:
            return datetime(year, mon, day, hh, mm)
        except ValueError:
            return None
    return None

In [ ]:
# [c04] pure_segment
# §4.1 (c) 分段，先切再洗（偏移對得回展示表）。D13：segment_no = 文件順序（上 → 下），時間順序看 segment_date。
# thread：有標頭區塊 → 每個區塊（含緊接在前的 -----Original Message / Appointment----- 行）起一段。
# 紀要：沒有標頭區塊 → 行首型號為界；`ITEM` 行後的 `m月d日` 是其後各段的日期。都切不開 → 整筆一段（segment_no 0）。
def _line_start(text: str, pos: int) -> int:
    return text.rfind("\n", 0, pos) + 1


def _boundary_before_block(text: str, block_start: int) -> int:
    """區塊起點所在行的行首；若前一個非空白行是 -----Original…----- 分隔線，再往前含該行。"""
    start = _line_start(text, block_start)
    prev = text[:start].rstrip()
    prev_line = prev[prev.rfind("\n") + 1:]
    if prev_line and ORIG_SEP_RE.match(prev_line):
        return prev.rfind("\n") + 1
    return start


def item_dates(text: str, mail_date: date | None) -> list[tuple[int, date]]:
    """`ITEM` 行之後 3 行內的 `m月d日` / `m/d` → [(ITEM 行位置, 日期)]。沒寫年 → mail_date 的年；
    推出的日期晚於 mail_date 超過 7 天就當成去年（跨年紀要）。"""
    out = []
    for m in ITEM_RE.finditer(text):
        tail = text[m.end():].lstrip("\n")
        window = "\n".join(tail.split("\n", 3)[:3])
        d = ITEM_DATE_RE.search(window)
        if not d:
            continue
        year = int(d.group(1)) if d.group(1) else (mail_date.year if mail_date else None)
        if year is None:
            continue
        try:
            dt = date(year, int(d.group(2)), int(d.group(3)))
        except ValueError:
            continue
        if not d.group(1) and mail_date and (dt - mail_date).days > 7:
            dt = dt.replace(year=year - 1)
        out.append((m.start(), dt))
    return out


def split_segments(display: str, mail_date: date | None) -> list[dict]:
    """回傳文件順序的段：[{segment_no, kind (thread / minutes / whole), start, end, block (標頭區塊或 None),
    segment_date, date_inferred}]。段 0 = 第一個界線前的文字（thread 的轉寄人附註、紀要的開場白）；
    是否保留由 [c05] 依長度決定。"""
    text = display or ""
    if not text.strip():
        return []
    blocks = find_header_blocks(text)
    segs = []
    if blocks:
        bounds = [_boundary_before_block(text, b["start"]) for b in blocks]
        if bounds[0] > 0:
            segs.append({"segment_no": 0, "kind": "thread", "start": 0, "end": bounds[0], "block": None})
        for n, b in enumerate(blocks, start=1):
            end = bounds[n] if n < len(bounds) else len(text)
            segs.append({"segment_no": n, "kind": "thread", "start": bounds[n - 1], "end": end, "block": b})
        for s in segs:
            sent = parse_datetime(s["block"]["fields"].get("Sent")) if s["block"] else None
            s["segment_date"], s["date_inferred"] = (sent.date(), False) if sent else (mail_date, True)
        return segs
    marks = [_line_start(text, m.start()) for m in MODEL_LINE_RE.finditer(text)]
    marks = sorted(set(marks))
    if not marks:
        return [{"segment_no": 0, "kind": "whole", "start": 0, "end": len(text), "block": None,
                 "segment_date": mail_date, "date_inferred": True}]
    dates = item_dates(text, mail_date)
    if marks[0] > 0:
        segs.append({"segment_no": 0, "kind": "minutes", "start": 0, "end": marks[0], "block": None})
    for n, st in enumerate(marks, start=1):
        end = marks[n] if n < len(marks) else len(text)
        segs.append({"segment_no": n, "kind": "minutes", "start": st, "end": end, "block": None})
    for s in segs:
        d = next((dt for pos, dt in reversed(dates) if pos < s["start"]), None)
        s["segment_date"], s["date_inferred"] = (d, False) if d else (mail_date, True)
    return segs

In [ ]:
# [c05] pure_clean
# 每段：規則 4（標頭改寫）→ 5（token 化）→ 6（短行攤平，D16）→ content_type / parties / hints → content_hash。
# 規則 7 簡轉繁：cluster 沒有 OpenCC，Phase 1 不做（D17），clean_text 保留原字；quote 以此為準。
def _name_variants(p: dict) -> set[str]:
    """一個參與者的所有寫法：原字串、中文名、英文名、「姓, 名」。"""
    out = {p.get("display_name_raw"), p.get("name_zh"), p.get("name_en")}
    en = norm_space(p.get("name_en"))
    words = en.split(" ")
    if len(words) >= 2:
        out.add(f"{words[-1]}, {' '.join(words[:-1])}")
    return {norm_space(s) for s in out if s and norm_space(s)}


def _is_ascii(s: str) -> bool:
    return all(ord(c) < 128 for c in s)


def build_token_map(parts: list[dict]) -> list[tuple[str, str]]:
    """參與者列 → [(要找的字串, 替換成什麼)]，長的在前。人：姓名各寫法與 email → <Pnn>；群組：只換 email → <Gnn>；
    轉寄人：帳號 → <P01>，工號 → 刪。ASCII 字串至少 3 字、中文至少 2 字，避免誤殺。"""
    pairs = {}
    for p in parts:
        tok = f"<{p['token']}>"
        if p.get("email"):
            pairs.setdefault(p["email"], tok)
        if p.get("kind") == "group":
            continue
        for s in _name_variants(p):
            if (_is_ascii(s) and len(s) >= 3) or (not _is_ascii(s) and len(s) >= 2):
                pairs.setdefault(s, tok)
        if p.get("account") and len(p["account"]) >= 3:
            pairs.setdefault(p["account"], tok)
        if p.get("emp_id"):
            pairs.setdefault(p["emp_id"], "")
    # 轉寄人原字串「工號/姓名/帳號/部門」整串不會出現在正文，拆開的各段上面已涵蓋
    return sorted(pairs.items(), key=lambda kv: (-len(kv[0]), kv[0]))


def first_name_patterns(parts: list[dict]) -> list[tuple[re.Pattern, str]]:
    """D21：簽名檔常寫「Alex 王小美」→ token 化後剩「Alex <P01>」。只對「英文名第一個字 ＋ 空白 ＋ 該人的 token」
    這個組合再換一次，不單獨換名字（Will、May 會誤殺）。"""
    firsts: dict[str, set[str]] = {}
    for p in parts:
        if p.get("kind") == "group" or not p.get("name_en"):
            continue
        first = norm_space(p["name_en"]).split(" ")[0].strip(",")
        if len(first) >= 2 and _is_ascii(first):
            firsts.setdefault(p["token"], set()).add(first)
    out = []
    for tok, names in sorted(firsts.items()):
        alt = "|".join(re.escape(n) for n in sorted(names))
        out.append((re.compile(rf"(?<![A-Za-z0-9])(?:{alt})\s+<{tok}>"), f"<{tok}>"))
    return out


def strip_urls(text: str) -> str:
    """D21：外連圖片 → [image]、裸網址 → [url]。L1 用不到網址，簽名檔的橫幅圖連結還會把段 0 撐過 150 字。"""
    return URL_RE.sub("[url]", MD_IMG_URL_RE.sub("[image]", text))


def _entry_pattern(s: str) -> str:
    """ASCII：前後不能接字母數字；單字（無空白）區分大小寫以免吃掉一般英文字（Will、May）；多字與 email 不分大小寫。
    中文：直接比對。用 inline flag 讓同一個 regex 混合兩種大小寫規則。"""
    esc = re.escape(s)
    if not _is_ascii(s):
        return esc
    body = f"(?<![A-Za-z0-9]){esc}(?![A-Za-z0-9])"
    return f"(?i:{body})" if " " in s or "," in s or "@" in s else body


def compile_token_regex(token_map: list[tuple[str, str]]):
    """一個大 alternation（長的在前）＋ 查表；回傳 (regex, lookup)。沒有任何項目 → (None, {})。"""
    if not token_map:
        return None, {}
    lookup = {k.lower(): v for k, v in token_map}
    rx = re.compile("|".join(_entry_pattern(k) for k, _ in token_map))
    return rx, lookup


def apply_tokens(text: str, rx, lookup: dict) -> str:
    if rx is None:
        return text
    return rx.sub(lambda m: lookup.get(m.group(0).lower(), ""), text)


def rewrite_header(block: dict, from_token: str | None, rx, lookup: dict) -> str:
    """規則 4：From → `<Pnn>:`；Sent 保留日期（yyyy-mm-dd hh:mm）；Subject 保留（會 token 化）；To / Cc / Bcc / When / Where 刪。"""
    f = block["fields"]
    who = from_token and f"<{from_token}>" or apply_tokens(f.get("From", ""), rx, lookup) or "<unknown>"
    lines = [f"{who}:"]
    sent = parse_datetime(f.get("Sent"))
    if sent:
        lines.append("Sent: " + sent.strftime("%Y-%m-%d %H:%M"))
    if f.get("Subject"):
        lines.append("Subject: " + apply_tokens(f["Subject"], rx, lookup))
    return "\n".join(lines)


def _is_junk_table_row(line: str) -> bool:
    """md 表格列，但每格都是空、分隔線（---）、純網址或 [image] → 版面殘渣（AUO e-news 橫幅、空表格）。"""
    s = line.strip()
    if not (s.startswith("|") and s.endswith("|") and len(s) >= 2):
        return False
    cells = [c.strip() for c in s[1:-1].split("|")]
    return all(not c or TABLE_SEP_CELL_RE.fullmatch(c) or TABLE_JUNK_CELL_RE.fullmatch(c) for c in cells)


def drop_junk_table_rows(text: str) -> str:
    """D21：刪掉只有版面、沒有內容的 md 表格列。Phase 1 跑出來段 0 幾乎都是這種 293 字的橫幅。"""
    return "\n".join(ln for ln in text.split("\n") if not _is_junk_table_row(ln))


def content_hash_of(text: str) -> str:
    """D21：token 編號是 pk 內的出現順序，同一段被不同信轉寄時 <P02> 可能變 <P05>；算 hash 前先把編號抹掉，
    不然跨信去重永遠對不到。"""
    return sha1_hex(normalize_text(TOKEN_NUM_RE.sub(r"<\1>", text)))


def flatten_short_lines(text: str) -> str:
    """規則 6（D16）：連續 ≥ 8 個短行（≤ 20 字；中間可夾空行，Word 表格攤平後每格一行）併成一行、以 ` | ` 分隔。
    不猜表頭（欄數未知），只做 deterministic 的壓縮。"""
    lines = text.split("\n")
    out, run, n_short = [], [], 0  # run = 連續短行（含夾在中間的空行）

    def flush():
        nonlocal n_short
        if n_short >= FLATTEN_MIN_LINES:
            out.append(" | ".join(ln.strip() for ln in run if ln.strip()))
            if run and not run[-1].strip():
                out.append("")
        else:
            out.extend(run)
        run.clear()
        n_short = 0

    for ln in lines:
        s = ln.strip()
        if s and len(s) <= SHORT_LINE_CHARS:
            run.append(ln)
            n_short += 1
        elif not s and run:
            run.append(ln)
        else:
            flush()
            out.append(ln)
    flush()
    t = "\n".join(out)
    t = re.sub(r"[ \t]+\n", "\n", t)
    return re.sub(r"\n{3,}", "\n\n", t).strip()


def classify(kind: str, text: str, clean_len: int, block: dict | None, attach_count: int) -> str:
    """§4.1 (d)。150–300 字又沒其他訊號：thread 段算 thread_reply，紀要段有型號算 inline_minutes，否則 noise（D18）。"""
    n_models = len(set(MODEL_RE.findall(text)))
    has_item = bool(ITEM_RE.search(text))
    fields = (block or {}).get("fields", {})
    if has_item or n_models >= 3 or (MINUTES_HINT_RE.search(text) and clean_len >= MINUTES_MIN_CHARS):
        return "inline_minutes"
    if kind == "thread" and block and clean_len >= THREAD_MIN_CHARS:
        return "thread_reply"
    if block and "When" in fields and clean_len < THREAD_MIN_CHARS:
        return "appointment"
    if clean_len < THREAD_MIN_CHARS and (attach_count > 0 or ATTACH_HINT_RE.search(text)):
        return "attachment_only"
    if clean_len < MIN_SEG_CHARS:
        return "noise"
    if kind == "thread" and block:
        return "thread_reply"
    return "inline_minutes" if n_models >= 1 else "noise"


def compile_alias_regex(aliases: list[tuple[str, str]]):
    """客戶別名 [(alias, entity_id)] → (regex 長的在前, lookup)。ASCII 別名要字邊界（VD、HP、LG）。"""
    if not aliases:
        return None, {}
    items = sorted({(a, e) for a, e in aliases if a}, key=lambda kv: (-len(kv[0]), kv[0]))
    lookup = {a.lower(): e for a, e in items}
    rx = re.compile("|".join(
        f"(?i:(?<![A-Za-z0-9]){re.escape(a)}(?![A-Za-z0-9]))" if _is_ascii(a) else re.escape(a) for a, _ in items
    ))
    return rx, lookup


def customer_hint(text: str, subject: str | None, rx, lookup: dict) -> str | None:
    """段內（＋主旨 ×3）命中最多的客戶 entity；長別名先吃掉（Samsung VD 不會再算成 Samsung）；平手 → NULL。"""
    if rx is None:
        return None
    score: dict[str, int] = {}
    for src, w in ((text, 1), (subject or "", 3)):
        for m in rx.finditer(src):
            e = lookup.get(m.group(0).lower())
            if e:
                score[e] = score.get(e, 0) + w
    if not score:
        return None
    best = sorted(score.items(), key=lambda kv: -kv[1])
    if len(best) > 1 and best[0][1] == best[1][1]:
        return None
    return best[0][0]


def _majority(votes: list[str]) -> str | None:
    if not votes:
        return None
    cnt: dict[str, int] = {}
    for v in votes:
        cnt[v] = cnt.get(v, 0) + 1
    best = sorted(cnt.items(), key=lambda kv: -kv[1])
    if len(best) > 1 and best[0][1] == best[1][1]:
        return None
    return best[0][0]


def app_hint(text: str, subject: str | None, customer: str | None, app_defaults: dict) -> str | None:
    """§6.4 優先序：型號前綴 → 尺寸語境 → 主旨關鍵字 → 客戶預設（只有一個 app 時）→ NULL。"""
    app = _majority([PREFIX_APP[m[0]] for m in MODEL_RE.findall(text)])
    if app:
        return app
    sizes = [m.group(1) for m in SIZE_HZ_RE.finditer(text)] or [m.group(1) for m in SIZE_RE.finditer(text)]
    votes = [a for s in sizes for a, S in (("MNT", MNT_SIZES), ("TV", TV_SIZES), ("NB", NB_SIZES)) if s in S]
    app = _majority(votes)
    if app:
        return app
    hits = [a for a, rx in SUBJECT_APP_RE.items() if rx.search(subject or "")]
    if len(hits) == 1:
        return hits[0]
    d = app_defaults.get(customer or "")  # s_semantic_entity.app_default 是 ARRAY<STRING>；也接受 "TV|MNT" 字串
    apps = [a for a in (list(d) if isinstance(d, (list, tuple)) else str(d or "").split("|")) if a]
    return apps[0] if len(apps) == 1 else None


def build_segment_rows(mail: dict, parts: list[dict], aliases: list[tuple[str, str]], app_defaults: dict,
                       cleaner_version: str) -> list[dict]:
    """展示表一列 ＋ 該 pk 的參與者列 → 清洗表多列（dup_of / cleaned_at 由 Spark 端填）。
    回傳的每列多兩個欄：`_residual` = token 化後仍殘留的參與者姓名數（硬規則）、`_masked` = 遮成 [email] 的數
    （關卡用，不入表）。"""
    display = mail.get("body_display") or ""
    segs = split_segments(display, mail.get("mail_date"))
    if not segs:
        return []
    token_map = build_token_map(parts)
    rx, lookup = compile_token_regex(token_map)
    first_pats = first_name_patterns(parts)
    arx, alookup = compile_alias_regex(aliases)
    from_tok = {p["segment_no"]: p["token"] for p in parts if p.get("role") == "from"}
    dom_by_seg: dict[int, set[str]] = {}
    for p in parts:
        if p.get("email_domain"):
            dom_by_seg.setdefault(p["segment_no"], set()).add(p["email_domain"].lower())
    multi = len(segs) > 1
    rows = []
    for s in segs:
        raw = display[s["start"]:s["end"]]
        if s["block"]:
            head = rewrite_header(s["block"], from_tok.get(s["segment_no"]), rx, lookup)
            body = display[s["block"]["end"]:s["end"]]
            text = head + "\n\n" + apply_tokens(body, rx, lookup)
        else:
            text = apply_tokens(raw, rx, lookup)
        # D20：參與者表沒有的 email（簽名檔、非標頭的收件列、IMCEAEX 尾巴）換不到 token → 遮成 [email]，計數不擋
        text, n_masked = EMAIL_ANY_RE.subn("[email]", text)
        for frx, tok in first_pats:
            text = frx.sub(tok, text)
        text = flatten_short_lines(drop_junk_table_rows(strip_urls(text)))
        clean_len = len(text)
        if multi and s["segment_no"] == 0 and clean_len < MIN_SEG_CHARS:
            continue  # 段 0（轉寄人附註 / 開場白）太短就不留（STATUS §6.2）
        found_domains = {m.split("@")[1].lower() for m in EMAIL_ANY_RE.findall(raw)}
        seg_domains = dom_by_seg.get(s["segment_no"] if s["block"] else 0, set())
        residual = len(rx.findall(text)) if rx else 0  # token 化後仍找得到參與者姓名 → 硬規則（理論上 0）
        cust = customer_hint(text, mail.get("subject_clean"), arx, alookup)
        rows.append({
            "pk_seq": mail["pk_seq"], "segment_no": s["segment_no"], "sub_no": 0,
            "drupal_id": mail.get("drupal_id"), "drupal_link": mail.get("drupal_link"), "unit": mail.get("unit"),
            "create_date": mail.get("create_date"), "mail_date": mail.get("mail_date"),
            "mail_datetime": mail.get("mail_datetime"),
            "segment_date": s["segment_date"], "date_inferred": bool(s["date_inferred"]),
            "subject_raw": mail.get("subject_raw"), "subject_clean": mail.get("subject_clean"),
            "forwarder_token": mail.get("forwarder_token"), "forwarder_dept_code": mail.get("forwarder_dept_code"),
            "meeting_when": mail.get("meeting_when"), "meeting_where": mail.get("meeting_where"),
            "parties": sorted(seg_domains | found_domains),
            "customer_hint": cust,
            "app_hint": app_hint(text, mail.get("subject_clean"), cust, app_defaults),
            "content_type": classify(s["kind"], text, clean_len, s["block"], int(mail.get("attach_count") or 0)),
            "body_source_field": mail.get("body_source_field"), "body_conflict": mail.get("body_conflict"),
            "display_start": s["start"], "display_end": s["end"],
            "clean_text": text or None, "raw_len": len(raw), "clean_len": clean_len,
            "is_fw": mail.get("is_fw"), "attach_count": mail.get("attach_count"),
            "content_hash": content_hash_of(text) if clean_len >= MIN_SEG_CHARS else None,
            "dup_of": None, "cleaner_version": cleaner_version, "cleaned_at": None,
            "_residual": residual, "_masked": n_masked,
        })
    return rows

In [ ]:
# [c06] spark_transforms
# Spark 接線。參與者表依 pk 聚成 array 跟 mail 列一起餵 UDF（一封信一次 Python 呼叫），回傳 array<struct>（清洗表欄位 ＋ _residual）。
# 別名表很小（數十列），collect 到 driver 當 closure；不 cache（serverless）。
def load_hint_refs(df_alias: DataFrame, df_entity: DataFrame) -> tuple[list[tuple[str, str]], dict]:
    """customer_hint 用的 (alias, entity_id)（name / sb_brand、side = customer）與 app_hint 用的 {entity_id: app_default}。"""
    ent = df_entity.where("is_active").select("entity_id", "side", "app_default")
    al = (
        df_alias.where(F.col("alias_type").isin("name", "sb_brand") & F.col("is_active"))
        .join(ent.where(F.col("side") == "customer").select("entity_id"), "entity_id")
        .select("alias", "entity_id").distinct().collect()
    )
    defaults = {r["entity_id"]: r["app_default"] for r in ent.collect() if r["app_default"]}
    return [(r["alias"], r["entity_id"]) for r in al], defaults


def transform_segments(df_mail: DataFrame, df_part: DataFrame, clean_schema: StructType,
                       aliases: list[tuple[str, str]], app_defaults: dict, cleaner_version: str) -> DataFrame:
    """mail 表形狀 ＋ participants 表形狀 → 清洗表形狀（dup_of / cleaned_at 仍 NULL）＋ `_residual` / `_masked`。"""
    out_schema = StructType(clean_schema.fields + [StructField("_residual", IntegerType()),
                                                   StructField("_masked", IntegerType())])
    parts = df_part.groupBy("pk_seq").agg(F.collect_list(F.struct(*PART_COLS)).alias("_parts"))
    joined = df_mail.select(*MAIL_COLS).join(parts, "pk_seq", "left")

    def _many(m, ps):
        rows = build_segment_rows(m.asDict(), [p.asDict() for p in (ps or [])], aliases, app_defaults, cleaner_version)
        return [tuple(r.get(f.name) for f in out_schema.fields) for r in rows]

    udf = F.udf(_many, ArrayType(out_schema))
    return joined.select(F.explode(udf(F.struct(*MAIL_COLS), F.col("_parts"))).alias("s")).select("s.*")


def with_dup_of(df_seg: DataFrame, df_existing: DataFrame) -> DataFrame:
    """§4.1 (e) 段層去重：同 content_hash 取 (create_date, pk_seq, segment_no, sub_no) 最早者為正本，其餘 dup_of 指向它。
    候選池 = 清洗表既有的正本段（dup_of IS NULL；排除本批 pk，本批會整組重建）＋ 本批。content_hash NULL 不參與。"""
    keys = ["pk_seq", "segment_no", "sub_no"]
    cols = keys + ["content_hash", "create_date"]
    batch = df_seg.select(*cols)
    pool = (
        df_existing.where(F.col("dup_of").isNull()).select(*cols)
        .join(batch.select("pk_seq").distinct(), "pk_seq", "left_anti")
        .unionByName(batch)
        .where(F.col("content_hash").isNotNull())
    )
    w = Window.partitionBy("content_hash").orderBy("create_date", "pk_seq", "segment_no", "sub_no")
    ref = F.concat_ws("-", "pk_seq", "segment_no", "sub_no")
    first = (
        pool.withColumn("_ref", ref).withColumn("_first", F.first("_ref").over(w))
        .select(*keys, "_ref", "_first")
    )
    return (
        df_seg.drop("dup_of").join(first, keys, "left")
        .withColumn("dup_of", F.when(F.col("_first") != F.col("_ref"), F.col("_first")))
        .select(*df_seg.columns)
    )

In [ ]:
# [c10] load_source
# 本批 = mail 表有、清洗表沒有的 pk ＋ mail 表比清洗表新（第一支重洗過）＋ 清洗表版本 != cleaner_version。不另設水位。
# 不分段的信直接排除（§3.0b / §4.1）：body 為 none、整封重複（dup_of_pk 非 NULL）、oversize。backfill 用 mail_date 窗。
def fq(name: str) -> str:
    return f"{cfg['catalog']}.{cfg['schema']}.{name}"


fq_mail, fq_part = fq(f"s_meeting_record_mail_{cfg['unit']}"), fq(f"s_meeting_participants_{cfg['unit']}")
fq_clean = fq(f"s_meeting_record_clean_{cfg['unit']}")
fq_alias, fq_entity = fq("s_semantic_entity_alias"), fq("s_semantic_entity")
for t in (fq_mail, fq_part, fq_clean, fq_alias, fq_entity):
    assert spark.catalog.tableExists(t), f"{t} 不存在：先跑 migration 001 / 002 與第一支 w1_t1_mail"

clean_tbl = spark.table(fq_clean)
mail_all = spark.table(fq_mail)
segmentable = (
    (F.col("body_source_field") != "none") & F.col("dup_of_pk").isNull()
    & ~F.coalesce(F.col("oversize"), F.lit(False)) & F.col("body_display").isNotNull()
)
if cfg["run_mode"] == "backfill":
    df_mail = mail_all.where(F.col("mail_date").between(F.lit(cfg["date_from"]), F.lit(cfg["date_to"])))
    scope = f"backfill mail_date {cfg['date_from']} ~ {cfg['date_to']}"
else:
    state = clean_tbl.groupBy("pk_seq").agg(
        F.max("cleaned_at").alias("_clean_at"),
        F.max((F.col("cleaner_version") != cfg["cleaner_version"]).cast("int")).alias("_stale"),
    )
    df_mail = mail_all.join(state, "pk_seq", "left").where(
        F.col("_clean_at").isNull() | (F.col("cleaned_at") > F.col("_clean_at")) | (F.col("_stale") == 1)
    ).drop("_clean_at", "_stale")
    scope = f"incremental：清洗表沒有 / 較舊 / 版本 != {cfg['cleaner_version']} 的 pk"
if cfg["sample_limit"] > 0:
    df_mail = df_mail.orderBy(F.desc("create_date")).limit(cfg["sample_limit"])
    scope += f"；sample_limit {cfg['sample_limit']}"
df_mail_seg = df_mail.where(segmentable)
df_part = spark.table(fq_part).join(df_mail_seg.select("pk_seq"), "pk_seq", "left_semi")
print(scope)

In [ ]:
# [c11] run_transforms
# 分段 / token 化 UDF → 段層去重。不 cache（serverless）：[c12] 每個 action 會重算 UDF，日量百封級可接受。
clean_schema = spark.table(fq_clean).schema
aliases, app_defaults = load_hint_refs(spark.table(fq_alias), spark.table(fq_entity))
print(f"customer aliases {len(aliases)}, app defaults {len(app_defaults)}")

# 本批 pk 先收到 driver 釘住（百封級、只有整數）：df_mail 的 incremental 條件是「mail 表 cleaned_at 比清洗表新」，
# 是 lazy 的；[c20] 寫表後 [c30] 再求值會變 0 筆，關卡全在空集合上過（2026-10-19 發現）。
batch_pk_list = [int(r[0]) for r in df_mail.select("pk_seq").collect()]
batch_pks = spark.createDataFrame([(p,) for p in batch_pk_list] or [], "pk_seq bigint")
batch_pks.createOrReplaceTempView("_w1_clean_batch_pks")  # 本批所有 pk（含不分段的），寫表時整組重建
df_mail_seg = df_mail_seg.join(batch_pks, "pk_seq", "left_semi")
print(f"batch pks pinned: {len(batch_pk_list)}")

df_seg = (
    with_dup_of(
        transform_segments(df_mail_seg, df_part, clean_schema, aliases, app_defaults, cfg["cleaner_version"]),
        clean_tbl,
    )
    .withColumn("cleaned_at", F.current_timestamp())
    .select(*clean_schema.fieldNames(), "_residual", "_masked")
)
df_seg.createOrReplaceTempView("_w1_clean_batch")
print("transforms defined (lazy)")


In [ ]:
# [c12] gates
# 關卡：殘留 email / 姓名 > 0（S3 硬規則）；dup_of > 80%（§4.1 (e)）→ 不符就停。本批 0 段只印不停（D34：沒新信是週跑常態）。
# 其餘只寫 log：content_type 分布、壓縮率、每封段數、超長段（> 12k）、segment_date 推測比例、hint 覆蓋率。
def pct(n, d) -> float | None:
    return round(n / d, 3) if d else None


m_in = df_mail.agg(
    F.count("*").alias("mails"),
    F.sum((F.col("body_source_field") == "none").cast("int")).alias("skip_none"),
    F.sum(F.col("dup_of_pk").isNotNull().cast("int")).alias("skip_dup_mail"),
    F.sum(F.coalesce(F.col("oversize"), F.lit(False)).cast("int")).alias("skip_oversize"),
).collect()[0].asDict()
s = df_seg.agg(
    F.count("*").alias("segments"),
    F.countDistinct("pk_seq").alias("mails_segmented"),
    F.sum("_residual").alias("residual_names"),
    F.sum("_masked").alias("emails_masked"),
    F.sum((F.col("_masked") > 0).cast("int")).alias("segments_with_masked_email"),
    F.sum(F.col("dup_of").isNotNull().cast("int")).alias("n_dup"),
    F.sum((F.col("clean_len") >= MIN_SEG_CHARS).cast("int")).alias("n_dedup_pool"),
    F.sum((F.col("clean_len") > SUBNO_CHARS).cast("int")).alias("n_over_12k"),
    F.sum(F.col("date_inferred").cast("int")).alias("n_date_inferred"),
    F.sum(F.col("customer_hint").isNotNull().cast("int")).alias("n_customer_hint"),
    F.sum(F.col("app_hint").isNotNull().cast("int")).alias("n_app_hint"),
    F.sum((F.col("segment_no") == 0).cast("int")).alias("n_seg0"),
    F.percentile_approx("clean_len", 0.5).alias("clean_len_p50"),
    F.percentile_approx("clean_len", 0.95).alias("clean_len_p95"),
    F.percentile_approx(F.col("clean_len") / F.greatest(F.col("raw_len"), F.lit(1)), 0.5).alias("compress_p50"),
).collect()[0].asDict()
n = s["segments"]
ctype = {r["content_type"]: r["c"] for r in
         df_seg.groupBy("content_type").agg(F.count("*").alias("c")).collect()}
extractable = sum(v for k, v in ctype.items() if k in ("inline_minutes", "thread_reply"))

log = {
    "scope": scope, "mails_in_batch": m_in["mails"],
    "skipped": {k: m_in[k] for k in ("skip_none", "skip_dup_mail", "skip_oversize")},
    "mails_segmented": s["mails_segmented"], "segments": n,
    "segments_per_mail": pct(n, s["mails_segmented"]), "seg0_kept": s["n_seg0"],
    "content_type": ctype, "extractable_segments": extractable,
    "dup_of_rate": pct(s["n_dup"], s["n_dedup_pool"]), "dup_segments": s["n_dup"],
    "clean_len_p50": s["clean_len_p50"], "clean_len_p95": s["clean_len_p95"], "compress_p50": s["compress_p50"],
    "over_12k": s["n_over_12k"], "date_inferred_rate": pct(s["n_date_inferred"], n),
    "customer_hint_rate": pct(s["n_customer_hint"], n), "app_hint_rate": pct(s["n_app_hint"], n),
    "emails_masked": s["emails_masked"], "segments_with_masked_email": s["segments_with_masked_email"],
    "residual_names": s["residual_names"],
}
print(json.dumps(log, ensure_ascii=False, default=str, indent=1))
if n == 0:
    print("本批 0 段：沒有新信，或 mail 表本批全是 none / 重複 / oversize（週跑正常；後面 cell 空跑）")
if (s["residual_names"] or 0) > 0:
    # 只印位置與殘留的字串形狀（字元遮成 x、長度保留），不印姓名
    print("殘留姓名的段（最多 10 段）：")
    df_seg.where("_residual > 0").select("pk_seq", "segment_no", "_residual", "clean_len").limit(10).show()
assert (s["residual_names"] or 0) == 0, "clean_text 殘留參與者姓名（S3 硬規則）：回頭查 [c05] token 化"
assert (log["dup_of_rate"] or 0) <= 0.80, "dup_of > 80%：分段太碎或來源重複入庫（規格 §4.1 (e) 關卡）"
if cfg["dry_run"]:
    print("\ndry_run = true：不寫表。樣本（token 化後前 200 字）：")
    df_seg.select("pk_seq", "segment_no", "content_type", "segment_date", "date_inferred", "clean_len",
                  "customer_hint", "app_hint", "dup_of", F.substring("clean_text", 1, 200).alias("clean_head")
                  ).orderBy("pk_seq", "segment_no").limit(12).show(truncate=80, vertical=True)

In [ ]:
# [c20] write_clean
# 本批 pk 的段整組重建：先 DELETE 該 pk 的舊段（重切後段數可能不同），再 append。
# 不用 MERGE：鍵 (pk_seq, segment_no, sub_no) 重切時整組會換，MERGE 留不住「消失的段」的刪除。
# 其他 pk 指向本批段的 dup_of 不動（§4.1 (e) 5：回溯仍有效）。
if not cfg["dry_run"]:
    spark.sql(f"DELETE FROM {fq_clean} WHERE pk_seq IN (SELECT pk_seq FROM _w1_clean_batch_pks)")
    spark.table("_w1_clean_batch").drop("_residual", "_masked").write.mode("append").saveAsTable(fq_clean)
    print("rebuilt segments for batch →", fq_clean)

In [ ]:
# [c30] check_target
# 寫入後驗證：本批 pk 的段都在表裡且版本正確；鍵無重複；clean_text 沒有 email 樣式（S3 硬規則，用 SQL 再查一次）；
# dup_of 指向的段存在。
if not cfg["dry_run"]:
    batch_pks = spark.table("_w1_clean_batch_pks")
    now = spark.table(fq_clean).join(batch_pks, "pk_seq", "left_semi")
    stale = now.where(F.col("cleaner_version") != cfg["cleaner_version"]).count()
    dup_keys = now.groupBy("pk_seq", "segment_no", "sub_no").count().where("count > 1").count()
    email_left = now.where(F.col("clean_text").rlike(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9\-]+\.[A-Za-z]{2,}")).count()
    all_rows = spark.table(fq_clean)
    dangling = (
        now.where(F.col("dup_of").isNotNull()).select(F.col("dup_of").alias("_ref"))
        .join(all_rows.select(F.concat_ws("-", "pk_seq", "segment_no", "sub_no").alias("_ref")), "_ref", "left_anti")
        .count()
    )
    print({"batch_segments_in_table": now.count(), "stale_cleaner_version": stale, "dup_keys": dup_keys,
           "email_left_in_clean_text": email_left, "dangling_dup_of": dangling,
           "table_total": all_rows.count()})
    assert stale == 0, "有本批 pk 的 cleaner_version 不是本次版本"
    assert dup_keys == 0, "清洗表鍵重複"
    assert email_left == 0, "clean_text 殘留 email（S3 硬規則）"
    if dangling:
        print("WARN dup_of 指向不存在的段（被指向的 pk 已重切）：", dangling)